# 🛒 Customer Segmentation Analysis Using K-Means Clustering

**Author:** Aldi Susilo  
**Date:** 2025  
**Tools:** Python, Pandas, Scikit-Learn, Matplotlib, Seaborn, Tableau-ready export  

---

## 📌 Project Overview

This project performs **customer segmentation** using the **K-Means Clustering** algorithm on a retail transaction dataset.
The goal is to identify distinct customer groups based on their purchasing behavior using the **RFM (Recency, Frequency, Monetary)** framework,
enabling targeted marketing strategies for each segment.

### 🎯 Business Objectives
1. Identify meaningful customer segments from transaction data
2. Profile each segment based on RFM characteristics
3. Generate actionable marketing recommendations per segment
4. Export clean data for Tableau/Power BI dashboard visualization

### 📊 Dataset
- **Source:** Synthetic retail transaction data (simulated for analysis)
- **Records:** ~1,000 customers
- **Features:** CustomerID, TransactionDate, Amount, Frequency, ProductCategory

## 📋 Table of Contents

1. [Import Libraries](#1-import-libraries)
2. [Data Generation & Loading](#2-data-generation--loading)
3. [Exploratory Data Analysis (EDA)](#3-exploratory-data-analysis-eda)
4. [RFM Feature Engineering](#4-rfm-feature-engineering)
5. [Data Preprocessing & Normalization](#5-data-preprocessing--normalization)
6. [Optimal K Selection (Elbow + Silhouette)](#6-optimal-k-selection-elbow--silhouette)
7. [K-Means Clustering](#7-k-means-clustering)
8. [Cluster Evaluation](#8-cluster-evaluation)
9. [Cluster Visualization](#9-cluster-visualization)
10. [Cluster Profiling & Business Insights](#10-cluster-profiling--business-insights)
11. [Marketing Recommendations](#11-marketing-recommendations)
12. [Export for Dashboard](#12-export-for-dashboard)

## 1. Import Libraries

In [ ]:
# ── Standard Libraries ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Visualization ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Machine Learning ────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

# ── Date & Utility ──────────────────────────────────────────────────────────
from datetime import datetime, timedelta
import random

# ── Plot Style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
PALETTE = ['#2563EB', '#16A34A', '#DC2626', '#D97706', '#7C3AED', '#0891B2']

print('✅ Libraries loaded successfully')
print(f'   numpy    : {np.__version__}')
print(f'   pandas   : {pd.__version__}')
print(f'   sklearn  : imported')

## 2. Data Generation & Loading

> 💡 **Note:** In this project we generate a realistic synthetic dataset that mimics real retail transactions.
> You can replace this section with `pd.read_csv('your_data.csv')` to use your own data.

In [ ]:
random.seed(42)
np.random.seed(42)

N_CUSTOMERS = 1000
SNAPSHOT_DATE = datetime(2025, 1, 1)

# ── Simulate 4 latent customer types ────────────────────────────────────────
segments = {
    'Champion':        {'n': 200, 'recency': (1,30),   'frequency': (15,40), 'monetary': (500,2000)},
    'Loyal':           {'n': 250, 'recency': (15,60),  'frequency': (8,20),  'monetary': (200,800)},
    'At Risk':         {'n': 300, 'recency': (60,180), 'frequency': (2,8),   'monetary': (50,300)},
    'Lost/Inactive':   {'n': 250, 'recency': (180,365),'frequency': (1,3),   'monetary': (10,100)},
}

records = []
cid = 1001
for seg_name, cfg in segments.items():
    for _ in range(cfg['n']):
        recency   = random.randint(*cfg['recency'])
        frequency = random.randint(*cfg['frequency'])
        monetary  = round(random.uniform(*cfg['monetary']), 2)
        last_purchase = SNAPSHOT_DATE - timedelta(days=recency)
        records.append({
            'CustomerID':    f'C{cid:04d}',
            'LastPurchase':  last_purchase.strftime('%Y-%m-%d'),
            'Frequency':     frequency,
            'TotalSpend':    monetary,
            'AvgOrderValue': round(monetary / frequency, 2),
            'true_segment':  seg_name,      # ground truth (dropped before modeling)
        })
        cid += 1

df_raw = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'✅ Dataset generated: {df_raw.shape[0]} customers × {df_raw.shape[1]} columns')
df_raw.head(10)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Basic Info ──────────────────────────────────────────────────────────────
print('='*55)
print('DATASET INFO')
print('='*55)
print(df_raw.info())
print()
print('='*55)
print('MISSING VALUES')
print('='*55)
print(df_raw.isnull().sum())
print()
print('='*55)
print('DESCRIPTIVE STATISTICS')
print('='*55)
df_raw[['Frequency','TotalSpend','AvgOrderValue']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Distribution of Key Customer Metrics', fontsize=14, fontweight='bold', y=1.02)

cols = [('Frequency', 'Purchase Frequency', PALETTE[0]),
        ('TotalSpend', 'Total Spend (IDR)', PALETTE[1]),
        ('AvgOrderValue', 'Avg Order Value (IDR)', PALETTE[3])]

for ax, (col, title, color) in zip(axes, cols):
    ax.hist(df_raw[col], bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    mean_val = df_raw[col].mean()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'Mean: {mean_val:.1f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('01_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Figure saved: 01_distribution.png')

In [ ]:
# ── Correlation Heatmap ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap
corr = df_raw[['Frequency','TotalSpend','AvgOrderValue']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', ax=axes[0],
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
axes[0].set_title('Correlation Matrix', fontweight='bold')

# Scatter
axes[1].scatter(df_raw['Frequency'], df_raw['TotalSpend'],
                alpha=0.4, color=PALETTE[0], s=20)
axes[1].set_xlabel('Purchase Frequency')
axes[1].set_ylabel('Total Spend (IDR)')
axes[1].set_title('Frequency vs Total Spend', fontweight='bold')

plt.tight_layout()
plt.savefig('02_correlation.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. RFM Feature Engineering

**RFM** is a proven framework to quantify customer value:

| Feature | Description | Business Meaning |
|---------|-------------|------------------|
| **Recency (R)** | Days since last purchase | How recently did the customer buy? |
| **Frequency (F)** | Number of transactions | How often do they buy? |
| **Monetary (M)** | Total amount spent | How much do they spend? |

In [ ]:
# ── Calculate Recency ───────────────────────────────────────────────────────
df_raw['LastPurchase'] = pd.to_datetime(df_raw['LastPurchase'])
df_raw['Recency'] = (SNAPSHOT_DATE - df_raw['LastPurchase']).dt.days

# ── Build RFM Table ──────────────────────────────────────────────────────────
rfm = df_raw[['CustomerID', 'Recency', 'Frequency', 'TotalSpend']].copy()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print('RFM Feature Summary:')
print('='*45)
print(rfm[['Recency','Frequency','Monetary']].describe().round(2))
print()
print(f'Total customers in RFM table: {len(rfm)}')
rfm.head()

In [ ]:
# ── RFM Distribution ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('RFM Feature Distributions', fontsize=14, fontweight='bold')

rfm_cols = [('Recency','Days Since Last Purchase', PALETTE[3]),
            ('Frequency','Number of Purchases', PALETTE[0]),
            ('Monetary','Total Spend (IDR)', PALETTE[1])]

for ax, (col, label, color) in zip(axes, rfm_cols):
    sns.histplot(rfm[col], bins=25, ax=ax, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(f'{col}\n({label})', fontweight='bold')
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig('03_rfm_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Data Preprocessing & Normalization

> K-Means uses Euclidean distance — **feature scaling is mandatory** to prevent high-magnitude features from dominating the clustering.
> We apply **StandardScaler** (zero mean, unit variance).

In [ ]:
# ── Log Transform (handle skewed monetary) ──────────────────────────────────
rfm['Recency_log']   = np.log1p(rfm['Recency'])
rfm['Frequency_log'] = np.log1p(rfm['Frequency'])
rfm['Monetary_log']  = np.log1p(rfm['Monetary'])

features = ['Recency_log', 'Frequency_log', 'Monetary_log']
X = rfm[features].values

# ── StandardScaler ───────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('✅ Preprocessing complete')
print(f'   Input shape  : {X.shape}')
print(f'   Scaled shape : {X_scaled.shape}')
print(f'   Mean after scaling : {X_scaled.mean(axis=0).round(4)}')
print(f'   Std  after scaling : {X_scaled.std(axis=0).round(4)}')

# ── Before / After comparison ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Before vs After Scaling', fontsize=13, fontweight='bold')

for i, (feat, label) in enumerate(zip(features, ['Recency (log)', 'Frequency (log)', 'Monetary (log)'])):
    axes[0, i].hist(rfm[feat], bins=25, color=PALETTE[i], alpha=0.8, edgecolor='white')
    axes[0, i].set_title(f'Raw: {label}', fontsize=10)

    axes[1, i].hist(X_scaled[:, i], bins=25, color=PALETTE[i+3], alpha=0.8, edgecolor='white')
    axes[1, i].set_title(f'Scaled: {label}', fontsize=10)

axes[0, 0].set_ylabel('Raw Features', fontweight='bold')
axes[1, 0].set_ylabel('Scaled Features', fontweight='bold')
plt.tight_layout()
plt.savefig('04_scaling.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Optimal K Selection (Elbow + Silhouette)

We use two complementary methods:
- **Elbow Method** — find where the Within-Cluster Sum of Squares (WCSS) stops decreasing sharply
- **Silhouette Score** — measures how well each point fits its cluster (higher = better, max 1.0)

In [ ]:
K_RANGE = range(2, 11)
inertias, silhouettes = [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Optimal K Selection', fontsize=14, fontweight='bold')

# Elbow
axes[0].plot(list(K_RANGE), inertias, 'o-', color=PALETTE[0], linewidth=2, markersize=7)
axes[0].axvline(4, color='red', linestyle='--', linewidth=1.5, label='Optimal K=4')
axes[0].set_xlabel('Number of Clusters (K)', fontsize=11)
axes[0].set_ylabel('WCSS (Inertia)', fontsize=11)
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].legend()

# Silhouette
colors = [PALETTE[1] if s == max(silhouettes) else PALETTE[0] for s in silhouettes]
bars = axes[1].bar(list(K_RANGE), silhouettes, color=colors, edgecolor='white')
axes[1].axvline(4, color='red', linestyle='--', linewidth=1.5, label='Optimal K=4')
axes[1].set_xlabel('Number of Clusters (K)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].legend()

for bar, score in zip(bars, silhouettes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{score:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('05_optimal_k.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'\n📊 Silhouette scores per K:')
for k, s in zip(K_RANGE, silhouettes):
    star = ' ← best' if s == max(silhouettes) else ''
    print(f'   K={k}: {s:.4f}{star}')

## 7. K-Means Clustering

In [ ]:
OPTIMAL_K = 4

kmeans = KMeans(
    n_clusters=OPTIMAL_K,
    init='k-means++',    # smarter initialization
    n_init=20,           # run 20 times, keep best
    max_iter=500,
    random_state=42
)

rfm['Cluster'] = kmeans.fit_predict(X_scaled)

print(f'✅ K-Means fitted with K={OPTIMAL_K}')
print(f'   Inertia (WCSS)    : {kmeans.inertia_:.2f}')
print(f'   Iterations        : {kmeans.n_iter_}')
print(f'   Final Silhouette  : {silhouette_score(X_scaled, rfm["Cluster"]):.4f}')
print()
print('Cluster distribution:')
print(rfm['Cluster'].value_counts().sort_index())

## 8. Cluster Evaluation

### Silhouette Analysis
Silhouette coefficient per sample: **close to +1 = well-clustered**, near 0 = on boundary, negative = possibly misclassified.

In [ ]:
sil_vals = silhouette_samples(X_scaled, rfm['Cluster'])
rfm['Silhouette'] = sil_vals

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10

for i in range(OPTIMAL_K):
    cluster_sil = np.sort(sil_vals[rfm['Cluster'] == i])
    size = len(cluster_sil)
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                     facecolor=PALETTE[i], edgecolor=PALETTE[i], alpha=0.8)
    ax.text(-0.05, y_lower + size/2, f'Cluster {i}', fontsize=10, color=PALETTE[i], fontweight='bold')
    y_lower = y_upper + 10

avg_score = silhouette_score(X_scaled, rfm['Cluster'])
ax.axvline(avg_score, color='red', linestyle='--', linewidth=1.5)
ax.text(avg_score + 0.01, 5, f'Avg: {avg_score:.3f}', color='red', fontsize=10)
ax.set_xlabel('Silhouette Coefficient', fontsize=11)
ax.set_ylabel('Cluster', fontsize=11)
ax.set_title('Silhouette Analysis per Cluster', fontsize=13, fontweight='bold')
ax.set_yticks([])
plt.tight_layout()
plt.savefig('06_silhouette.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'\n📊 Silhouette Score Summary by Cluster:')
print(rfm.groupby('Cluster')['Silhouette'].agg(['mean','min','max']).round(4))

## 9. Cluster Visualization

In [ ]:
# ── PCA 2D Projection ────────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
rfm['PCA1'] = X_pca[:, 0]
rfm['PCA2'] = X_pca[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Cluster Visualization', fontsize=14, fontweight='bold')

# PCA scatter
for i in range(OPTIMAL_K):
    mask = rfm['Cluster'] == i
    axes[0].scatter(rfm.loc[mask,'PCA1'], rfm.loc[mask,'PCA2'],
                    label=f'Cluster {i}', color=PALETTE[i], alpha=0.6, s=25)

# centroids in PCA space
centers_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centers_pca[:,0], centers_pca[:,1],
                c='black', marker='X', s=180, zorder=5, label='Centroids')
axes[0].set_title('PCA 2D Projection', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].legend()

# Recency vs Monetary
for i in range(OPTIMAL_K):
    mask = rfm['Cluster'] == i
    axes[1].scatter(rfm.loc[mask,'Recency'], rfm.loc[mask,'Monetary'],
                    label=f'Cluster {i}', color=PALETTE[i], alpha=0.6, s=25)
axes[1].set_title('Recency vs Monetary', fontweight='bold')
axes[1].set_xlabel('Recency (days)')
axes[1].set_ylabel('Monetary (IDR)')
axes[1].legend()

plt.tight_layout()
plt.savefig('07_cluster_viz.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}% of total variance')

In [ ]:
# ── Radar Chart per Cluster ───────────────────────────────────────────────────
from matplotlib.patches import FancyArrowPatch

profile = rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean()

# Normalize 0-1 for radar (invert Recency: lower=better)
normed = profile.copy()
normed['Recency']   = 1 - (profile['Recency']   - profile['Recency'].min())   / (profile['Recency'].max()   - profile['Recency'].min())
normed['Frequency'] =     (profile['Frequency'] - profile['Frequency'].min()) / (profile['Frequency'].max() - profile['Frequency'].min())
normed['Monetary']  =     (profile['Monetary']  - profile['Monetary'].min())  / (profile['Monetary'].max()  - profile['Monetary'].min())

labels  = ['Recency\n(inverted)', 'Frequency', 'Monetary']
N       = len(labels)
angles  = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, OPTIMAL_K, figsize=(16, 4), subplot_kw=dict(polar=True))
fig.suptitle('Cluster Radar Profiles (Normalized RFM)', fontsize=13, fontweight='bold')

for i, ax in enumerate(axes):
    vals = normed.iloc[i].tolist() + normed.iloc[i].tolist()[:1]
    ax.plot(angles, vals, color=PALETTE[i], linewidth=2)
    ax.fill(angles, vals, color=PALETTE[i], alpha=0.3)
    ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title(f'Cluster {i}', fontweight='bold', color=PALETTE[i], pad=15)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('08_radar.png', bbox_inches='tight', dpi=150)
plt.show()

## 10. Cluster Profiling & Business Insights

Map each numeric cluster to a meaningful business segment label.

In [ ]:
# ── Compute cluster means ────────────────────────────────────────────────────
summary = rfm.groupby('Cluster').agg(
    Count       = ('CustomerID', 'count'),
    Recency_avg = ('Recency', 'mean'),
    Freq_avg    = ('Frequency', 'mean'),
    Monetary_avg= ('Monetary', 'mean'),
    Sil_avg     = ('Silhouette', 'mean')
).round(2)

summary['% Customers'] = (summary['Count'] / summary['Count'].sum() * 100).round(1)
print(summary.to_string())

# ── Assign segment labels ────────────────────────────────────────────────────
# Sort clusters by combined score (low recency, high freq, high monetary)
summary['score'] = (-summary['Recency_avg'] + summary['Freq_avg'] + summary['Monetary_avg']/100)
rank = summary['score'].rank(ascending=False).astype(int)

SEGMENT_NAMES = {
    1: 'Champions',
    2: 'Loyal Customers',
    3: 'At Risk',
    4: 'Lost / Inactive'
}
cluster_label_map = {cluster: SEGMENT_NAMES[r] for cluster, r in rank.items()}
rfm['Segment'] = rfm['Cluster'].map(cluster_label_map)

print('\n🏷️  Cluster → Segment mapping:')
for c, s in cluster_label_map.items():
    print(f'   Cluster {c} → {s}')

In [ ]:
# ── Segment summary bar charts ───────────────────────────────────────────────
seg_order  = ['Champions', 'Loyal Customers', 'At Risk', 'Lost / Inactive']
seg_colors = dict(zip(seg_order, PALETTE[:4]))

seg_summary = rfm.groupby('Segment').agg(
    Count       = ('CustomerID', 'count'),
    Recency_avg = ('Recency', 'mean'),
    Freq_avg    = ('Frequency', 'mean'),
    Monetary_avg= ('Monetary', 'mean'),
).reindex(seg_order).round(2)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Segment Profiles', fontsize=14, fontweight='bold')

metrics = [
    ('Count',        '# Customers',         axes[0,0]),
    ('Recency_avg',  'Avg Recency (days)',   axes[0,1]),
    ('Freq_avg',     'Avg Frequency',        axes[1,0]),
    ('Monetary_avg', 'Avg Total Spend (IDR)',axes[1,1]),
]

for col, ylabel, ax in metrics:
    colors_list = [seg_colors[s] for s in seg_summary.index]
    bars = ax.bar(seg_summary.index, seg_summary[col], color=colors_list, edgecolor='white')
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(ylabel, fontweight='bold', fontsize=10)
    ax.tick_params(axis='x', rotation=20)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h * 1.01,
                f'{h:,.0f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('09_segment_profile.png', bbox_inches='tight', dpi=150)
plt.show()

## 11. Marketing Recommendations

| Segment | Size | Recency | Frequency | Monetary | Strategy |
|---------|------|---------|-----------|----------|----------|
| 🏆 **Champions** | ~20% | Very recent | Very high | Very high | Reward & retain: loyalty program, early access, VIP perks |
| 💛 **Loyal Customers** | ~25% | Recent | High | Medium–High | Upsell & cross-sell: personalized recommendations, bundle offers |
| ⚠️ **At Risk** | ~30% | Medium | Low | Low–Medium | Win-back: re-engagement emails, time-limited discounts |
| ❌ **Lost / Inactive** | ~25% | Very old | Very low | Very low | Reactivation or sunset: last-chance campaign or remove from active list |

---

### Key Business Insights

1. **Champions** generate disproportionate revenue → protect them with exclusive benefits
2. **At Risk** is the largest segment → highest ROI opportunity for re-engagement campaigns
3. **Loyal Customers** are the best upsell target → already trust the brand
4. **Lost/Inactive** → evaluate cost-benefit of reactivation vs. removing from mailing list

In [ ]:
# ── Revenue contribution per segment ────────────────────────────────────────
rev = rfm.groupby('Segment')['Monetary'].sum().reindex(seg_order)
pct = (rev / rev.sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Revenue Contribution by Segment', fontsize=13, fontweight='bold')

# Pie
colors_list = [seg_colors[s] for s in seg_order]
axes[0].pie(rev, labels=seg_order, colors=colors_list, autopct='%1.1f%%',
            startangle=140, pctdistance=0.75,
            wedgeprops=dict(linewidth=1, edgecolor='white'))
axes[0].set_title('Revenue Share', fontweight='bold')

# Bar
bars = axes[1].bar(seg_order, rev.values, color=colors_list, edgecolor='white')
axes[1].set_ylabel('Total Revenue (IDR)', fontsize=10)
axes[1].set_title('Total Revenue per Segment', fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)
for bar, v in zip(bars, rev.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f'IDR {v:,.0f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('10_revenue_by_segment.png', bbox_inches='tight', dpi=150)
plt.show()

print('Revenue Summary:')
for seg in seg_order:
    print(f'   {seg:20s}: IDR {rev[seg]:>10,.0f}  ({pct[seg]}%)')

## 12. Export for Dashboard

Export the final enriched dataset to CSV for use in **Tableau** or **Power BI**.

In [ ]:
# ── Final export dataframe ───────────────────────────────────────────────────
export_df = rfm[['CustomerID','Recency','Frequency','Monetary','Cluster','Segment','Silhouette']].copy()
export_df['AvgOrderValue'] = df_raw['AvgOrderValue'].values
export_df.rename(columns={
    'Recency':   'Recency_days',
    'Frequency': 'Purchase_count',
    'Monetary':  'Total_spend_IDR'
}, inplace=True)

# Save CSV
export_df.to_csv('customer_segments_export.csv', index=False)

print('✅ Exported: customer_segments_export.csv')
print(f'   Rows    : {len(export_df)}')
print(f'   Columns : {list(export_df.columns)}')
print()
print('Final dataset preview:')
export_df.head(10)

In [ ]:
# ── Final Summary ────────────────────────────────────────────────────────────
print('=' * 55)
print('           PROJECT SUMMARY')
print('=' * 55)
print(f'  Algorithm     : K-Means Clustering (k-means++)')
print(f'  Optimal K     : {OPTIMAL_K}')
print(f'  Inertia       : {kmeans.inertia_:.2f}')
print(f'  Silhouette    : {silhouette_score(X_scaled, rfm["Cluster"]):.4f}')
print(f'  Total customers: {len(rfm)}')
print()
print('  Segment breakdown:')
for seg in seg_order:
    n = len(rfm[rfm['Segment']==seg])
    pct_n = n/len(rfm)*100
    print(f'    {seg:20s}: {n:4d} customers ({pct_n:.1f}%)')
print()
print('  Output files generated:')
outputs = [
    '01_distribution.png', '02_correlation.png', '03_rfm_distribution.png',
    '04_scaling.png', '05_optimal_k.png', '06_silhouette.png',
    '07_cluster_viz.png', '08_radar.png', '09_segment_profile.png',
    '10_revenue_by_segment.png', 'customer_segments_export.csv'
]
for o in outputs:
    print(f'    📄 {o}')
print('=' * 55)